# 01 - Why Geometric Algebra?

This is the canonical Chapter 1 notebook for *Geometric Algebra for Computer Science*.
It is standalone study material: the prose is original, the examples are executable, and
the plots are generated from code instead of copied from the book.

The opening chapter has a simple job. It shows that geometry programming becomes
fragile when every object carries its own special representation, then points toward a
single algebra where points, lines, planes, circles, spheres, and transformations can be
handled as related objects. The notebook uses NumPy coordinate formulas as a visible
shadow of that idea. Full geometric algebra arrives later; here we build the intuition and
check the geometry numerically.

Source coverage: the requested chapter span is printed pp. 1-22. In this PDF extraction,
Chapter 1 text appears on printed pp. 1-19, followed by Chapter 2 at printed p. 23.


## Translation Guide

| Chapter idea | Full geometric-algebra reading | Coordinate stand-in used here |
|---|---|---|
| Point | Modeled point element, later especially in homogeneous or conformal models | NumPy array `[x, y, z]` |
| Circle from three points | One object built from point elements | Euclidean circle fitted through `c1`, `c2`, and `c3` |
| Line from two points | A subspace/object with orientation and location | A point plus a unit direction |
| Plane reflector | Plane object used as an operator | Plane point plus unit normal |
| Rotation around a line | Rotor-like sandwich action on the whole object | Rotation matrix around a 3-D axis line |
| Reflection in a plane | Object/operator action on points, lines, and circles | Signed-distance reflection formula |
| Reflection in a sphere | Conformal sphere operator | Classical sphere inversion of sampled points |
| Interpolation | Operator parameter changes smoothly | Partial rotations sampled from angle `0` to `phi` |

Read each variable first as a geometric object and only second as coordinates. That is the
central habit Chapter 1 is trying to install.


### What the opening chapter is really changing

The first chapter is deliberately concrete: it does not ask you to accept a large formal algebra before seeing why such an algebra is useful.  The central move is to treat geometric objects as things that can be *constructed*, *combined*, and *acted on* without converting every step back into coordinates.  A line, a circle, and a plane are not just sets of points; they are computational objects with incidence tests, distances, orientations, and transformations attached to them.

In ordinary vector algebra the same geometric scene usually fragments into several unrelated tools.  Lines may be represented by point-direction pairs, planes by normal-distance equations, rotations by matrices or quaternions, and intersections by solving linear systems.  The chapter's promise is not that those older tools disappear, but that geometric algebra gives them a shared grammar.  The notebook therefore keeps asking one question: what information must an object carry so later operations can use it without re-deriving the whole scene from scratch?

A useful way to read the examples below is to separate three layers.  The visible layer is the figure.  The coordinate layer is the small amount of numerical data used to draw it.  The geometric layer is the invariant statement: three noncollinear points determine an oriented circle, a rotor preserves distances and orientation in its plane, a plane reflection preserves the mirror plane and reverses the signed normal direction, and sphere inversion swaps near and far according to a reciprocal distance rule.  The rest of the book makes the geometric layer the primary layer.


## Notebook Route

1. Build a chapter route map from the printed-page structure.
2. Set up reusable geometry helpers and artifact saving.
3. Construct the motivating scene from point handles.
4. Work the circle construction numerically.
5. Animate the line rotation and plane reflection as object-level actions.
6. Swap the plane for a sphere and study inversion.
7. Run sanity checks that verify the claims made by the plots.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

from IPython.display import Markdown
import numpy as np
import plotly.graph_objects as go

# Find the project root even when the notebook is opened from this chapter folder.
BOOK_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils" / "artifacts.py").exists():
        BOOK_ROOT = candidate
        break

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import save_plotly_html
from utils.chapter01_geometry import (
    COLORS,
    Circle3D,
    Line3D,
    Plane3D,
    Sphere3D,
    circle_point_errors,
    curve_trace,
    finish_figure,
    line_trace,
    plane_surface_trace,
    point_trace,
    rotate_points_about_line,
    segment_trace,
    sphere_surface_trace,
)

np.set_printoptions(precision=4, suppress=True)
ARTIFACT_ROOT = BOOK_ROOT / "artifacts"
PLOT_ROOT = ARTIFACT_ROOT / "chapter-01" / "plots"
PLOT_ROOT.mkdir(parents=True, exist_ok=True)

BOOK_ROOT


WindowsPath('D:/Geometry/Geometric-Algebra-for-Computer-Science')

## Chapter Route Map

Chapter 1 starts with a concrete construction, then backs out to the larger design
argument. The sections move from a visual example, to vector spaces as modeling tools,
to subspaces as computational objects, to transformations, interpolation, and finally the
programming route through the rest of the book.

The first artifact is a compact navigation map. Hover each bar to see what role that
section plays in this notebook.


In [2]:
route_rows = [
    ("1.1 Example", 1, 7, "Circle, line, rotation, plane reflection, then sphere inversion."),
    ("1.2.1 Models", 8, 1, "Vector spaces are useful models, but each model chooses what is easy."),
    ("1.2.2 Subspaces", 9, 3, "Lines, planes, and higher-dimensional spans become elements of computation."),
    ("1.2.3-1.2.4 Transforms", 12, 2, "Linear and orthogonal transformations extend beyond single vectors."),
    ("1.2.5 Operators", 14, 1, "Objects can also act as operators through sandwich-style formulas."),
    ("1.2.6 Interpolation", 15, 1, "Closed-form operator families support smooth motion and perturbation."),
    ("1.3 Programming", 15, 2, "A unified representation reduces conversion code and special cases."),
    ("1.4-1.5 Book route", 17, 3, "The book moves from products, to models, to implementation."),
]
labels = [row[0] for row in route_rows]
starts = [row[1] for row in route_rows]
durations = [row[2] for row in route_rows]
notes = [row[3] for row in route_rows]
colors = ["#4c78a8", "#59a14f", "#59a14f", "#f28e2b", "#b07aa1", "#e15759", "#76b7b2", "#9c755f"]

fig_route = go.Figure(
    go.Bar(
        x=durations,
        y=labels,
        base=starts,
        orientation="h",
        marker={"color": colors},
        customdata=np.column_stack([starts, np.array(starts) + np.array(durations) - 1, notes]),
        hovertemplate="%{y}<br>printed pp. %{customdata[0]}-%{customdata[1]}<br>%{customdata[2]}<extra></extra>",
    )
)
fig_route.update_layout(
    title="Chapter 1 route by printed page span",
    width=1000,
    height=460,
    xaxis={"title": "printed page", "range": [0.5, 20.5], "dtick": 1},
    yaxis={"autorange": "reversed"},
    margin={"l": 150, "r": 30, "t": 60, "b": 40},
    showlegend=False,
)

route_path = save_plotly_html(
    fig_route,
    "chapter-01",
    "plots",
    "chapter-route-map.html",
    root=ARTIFACT_ROOT,
)
print(f"saved {route_path}")
fig_route


saved D:\Geometry\Geometric-Algebra-for-Computer-Science\artifacts\chapter-01\plots\chapter-route-map.html


## The Motivating Scene

The chapter's visual seed can be recreated from a few handles:

- three points determine a circle;
- two points determine a line;
- one point and one normal determine a plane;
- a sphere later replaces the plane to show that a similar operator-shaped idea can create
  a different geometric effect.

In the book, these are geometric-algebra elements. Here they are coordinate objects with
the same visible behavior for the examples we need.


In [3]:
# Scene inputs: these are the coordinate handles for the notebook version of the chapter scene.
c1 = np.array([1.10, 0.95, 0.18])
c2 = np.array([-0.82, 0.62, -0.10])
c3 = np.array([0.20, -0.92, 0.55])
l1 = np.array([-1.25, -0.90, -0.55])
l2 = np.array([1.20, 0.70, 0.82])
plane_point = np.array([0.08, -0.12, 0.05])
plane_normal = np.array([0.70, -0.38, 0.61])

C = Circle3D.through_three_points(c1, c2, c3)
L = Line3D.through(l1, l2)
plane = Plane3D(plane_point, plane_normal)

circle_points = C.sample(180)
control_points = np.array([c1, c2, c3, l1, l2, plane_point])

print(f"circle center: {C.center}, radius: {C.radius:.4f}")
print(f"line direction: {L.direction}")
print(f"plane normal: {plane.normal}")


circle center: [0.2138 0.1974 0.2262], radius: 1.1635
line direction: [0.7583 0.4952 0.424 ]
plane normal: [ 0.6977 -0.3788  0.608 ]


In [4]:
fig_construct = go.Figure()
fig_construct.add_trace(plane_surface_trace(plane, "reflecting plane"))
fig_construct.add_trace(curve_trace(circle_points, "circle through c1,c2,c3", COLORS["circle"], width=8))
fig_construct.add_trace(line_trace(L, "line through l1,l2", COLORS["line"]))
fig_construct.add_trace(segment_trace(np.array([c1, c2, c3, c1]), "three point circle handles", "#2ca02c", width=3, dash="dot"))
fig_construct.add_trace(segment_trace(np.array([l1, l2]), "two point line handles", "#d62728", width=5))
fig_construct.add_trace(point_trace(control_points, ["c1", "c2", "c3", "l1", "l2", "p"], "#111111"))
fig_construct.add_trace(
    go.Cone(
        x=[plane_point[0]],
        y=[plane_point[1]],
        z=[plane_point[2]],
        u=[plane.normal[0]],
        v=[plane.normal[1]],
        w=[plane.normal[2]],
        name="plane normal",
        anchor="tail",
        sizemode="absolute",
        sizeref=0.35,
        colorscale=[[0, "#f2c14e"], [1, "#f2c14e"]],
        showscale=False,
    )
)
finish_figure(fig_construct, "Chapter 1 construction handles")

construction_path = save_plotly_html(
    fig_construct,
    "chapter-01",
    "plots",
    "construction-handles.html",
    root=ARTIFACT_ROOT,
)
print(f"saved {construction_path}")
fig_construct


saved D:\Geometry\Geometric-Algebra-for-Computer-Science\artifacts\chapter-01\plots\construction-handles.html


## Worked Example 1: Three Points Become One Circle

The first construction is deliberately modest. A circle is not stored as three unrelated
points; the points define one object with a center, radius, plane, and orientation. The
helper `Circle3D.through_three_points` computes those derived quantities, and the check
below verifies that the original handles really lie on the resulting circle.


In [5]:
handle_points = np.array([c1, c2, c3])
radial_errors, plane_errors = circle_point_errors(C, handle_points)
distances = np.linalg.norm(handle_points - C.center, axis=1)
area_vector = 0.5 * np.cross(c2 - c1, c3 - c1)

rows = ["| handle | distance to center | radial error | plane error |", "|---|---:|---:|---:|"]
for label, distance, radial, plane_err in zip(["c1", "c2", "c3"], distances, radial_errors, plane_errors):
    rows.append(f"| `{label}` | {distance:.6f} | {radial:.2e} | {plane_err:.2e} |")
rows.append("")
rows.append(f"The oriented triangle area magnitude is `{np.linalg.norm(area_vector):.6f}`; a zero value would mean the three points were collinear and no unique circle would be available.")
Markdown("\n".join(rows))


| handle | distance to center | radial error | plane error |
|---|---:|---:|---:|
| `c1` | 1.163508 | 0.00e+00 | -1.72e-17 |
| `c2` | 1.163508 | -2.22e-16 | 5.56e-17 |
| `c3` | 1.163508 | -4.44e-16 | 1.54e-16 |

The oriented triangle area magnitude is `1.745682`; a zero value would mean the three points were collinear and no unique circle would be available.

## Rotation and Plane Reflection

The chapter's main example then treats the circle and line as objects acted on by
transformations. In full geometric algebra, a rotor-like element can wrap the object in a
sandwich expression. This notebook uses the familiar 3-D rotation matrix around the line
`L`, but the important reading is the same: the whole circle moves, not just a single point.

The second artifact has a slider. Move it to see the circle rotate from angle `0` to `phi`,
while the plane-reflected version updates in step.


In [6]:
phi = np.pi / 2.0
alpha_values = np.linspace(0.0, 1.0, 13)
rotated_circle_points = rotate_points_about_line(circle_points, L, phi)
reflected_line = plane.reflect_line(L)
reflected_circle_points = plane.reflect_points(circle_points)
reflected_rotated_points = plane.reflect_points(rotated_circle_points)

def rotated_at(alpha: float) -> np.ndarray:
    return rotate_points_about_line(circle_points, L, alpha * phi)

fig_plane = go.Figure()
fig_plane.add_trace(plane_surface_trace(plane, "reflecting plane"))
fig_plane.add_trace(line_trace(L, "L", COLORS["line"]))
fig_plane.add_trace(line_trace(reflected_line, "plane reflection of L", COLORS["reflected"], width=5))
fig_plane.add_trace(curve_trace(circle_points, "C", COLORS["circle"], width=8))
fig_plane.add_trace(curve_trace(rotated_at(0.0), "rotating C", COLORS["rotated"], width=8))
fig_plane.add_trace(curve_trace(plane.reflect_points(rotated_at(0.0)), "plane reflection of rotating C", "#17becf", width=6))
fig_plane.add_trace(curve_trace(rotated_circle_points, "target at phi", "rgba(31,119,180,0.25)", width=4))
fig_plane.add_trace(curve_trace(reflected_circle_points, "plane reflection of original C", COLORS["reflected"], width=5))
fig_plane.add_trace(point_trace(np.array([c1, c2, c3, l1, l2]), ["c1", "c2", "c3", "l1", "l2"], "#111111"))

frames = []
for alpha in alpha_values:
    rotated = rotated_at(alpha)
    frames.append(
        go.Frame(
            data=[
                curve_trace(rotated, "rotating C", COLORS["rotated"], width=8),
                curve_trace(plane.reflect_points(rotated), "plane reflection of rotating C", "#17becf", width=6),
            ],
            traces=[4, 5],
            name=f"{alpha:.2f}",
        )
    )
fig_plane.frames = frames
fig_plane.update_layout(
    sliders=[
        {
            "active": 0,
            "currentvalue": {"prefix": "rotation fraction: "},
            "pad": {"t": 35},
            "steps": [
                {
                    "label": frame.name,
                    "method": "animate",
                    "args": [[frame.name], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}],
                }
                for frame in frames
            ],
        }
    ],
    updatemenus=[
        {
            "type": "buttons",
            "direction": "left",
            "x": 0.02,
            "y": 0.02,
            "buttons": [
                {
                    "label": "play",
                    "method": "animate",
                    "args": [None, {"frame": {"duration": 220, "redraw": True}, "fromcurrent": True}],
                },
                {
                    "label": "pause",
                    "method": "animate",
                    "args": [[None], {"mode": "immediate", "frame": {"duration": 0, "redraw": False}}],
                },
            ],
        }
    ],
)
finish_figure(fig_plane, "Chapter 1 object action: rotation and plane reflection")

plane_path = save_plotly_html(
    fig_plane,
    "chapter-01",
    "plots",
    "line-circle-plane-reflection.html",
    root=ARTIFACT_ROOT,
)
print(f"saved {plane_path}")
fig_plane


saved D:\Geometry\Geometric-Algebra-for-Computer-Science\artifacts\chapter-01\plots\line-circle-plane-reflection.html


## Worked Example 2: What Reflection Preserves and Reverses

A plane reflection reverses signed distance to the plane. It also preserves ordinary
lengths and is its own inverse: reflect twice and you get back what you started with. That
is the coordinate version of the chapter's broader claim that a geometric object can also
serve as a transformation operator.


In [7]:
example_points = np.array([c1, c2, c3, L.point_at(-0.6), L.point_at(0.8)])
reflected_examples = plane.reflect_points(example_points)
restored_examples = plane.reflect_points(reflected_examples)

before = plane.signed_distance(example_points)
after = plane.signed_distance(reflected_examples)
restore_error = np.max(np.linalg.norm(restored_examples - example_points, axis=1))

rows = ["| point | signed distance before | signed distance after | sum |", "|---|---:|---:|---:|"]
for i, (d0, d1) in enumerate(zip(before, after), start=1):
    rows.append(f"| `x{i}` | {d0:.6f} | {d1:.6f} | {d0 + d1:.2e} |")
rows.append("")
rows.append(f"Reflecting twice returns the sample points with max error `{restore_error:.2e}`.")
Markdown("\n".join(rows))


| point | signed distance before | signed distance after | sum |
|---|---:|---:|---:|
| `x1` | 0.385449 | -0.385449 | 1.11e-16 |
| `x2` | -0.999457 | 0.999457 | -6.66e-16 |
| `x3` | 0.690759 | -0.690759 | 3.33e-16 |
| `x4` | -1.356958 | 1.356958 | -4.44e-16 |
| `x5` | -0.517905 | 0.517905 | -1.11e-16 |

Reflecting twice returns the sample points with max error `1.28e-15`.

## Sphere Inversion Variant

The book's opening example also swaps the plane for a sphere. In the conformal model,
planes and spheres are part of one family of objects; here we use the classical coordinate
formula for sphere inversion. A point `x` is moved along the ray from the sphere center
`c` so that

$$
||x-c|| \, ||x'-c|| = r^2.
$$

This is a useful contrast: plane reflection keeps lines as lines, while sphere inversion can
bend a line into a circle-like curve when the line misses the sphere center.


In [8]:
sphere = Sphere3D(center=np.array([0.62, -0.18, 0.12]), radius=1.12)
line_points = L.sample(-3.0, 3.0, 240)
inverted_line_points = sphere.invert_points(line_points)
inverted_circle_points = sphere.invert_points(circle_points)
inverted_rotated_points = sphere.invert_points(rotated_circle_points)

fig_sphere = go.Figure()
fig_sphere.add_trace(sphere_surface_trace(sphere, "inversion sphere"))
fig_sphere.add_trace(line_trace(L, "L", COLORS["line"], t_min=-3.0, t_max=3.0))
fig_sphere.add_trace(curve_trace(circle_points, "C", COLORS["circle"], width=8))
fig_sphere.add_trace(curve_trace(rotated_circle_points, "rotated C", COLORS["rotated"], width=7))
fig_sphere.add_trace(curve_trace(inverted_line_points, "sphere inversion of L", COLORS["reflected"], width=6))
fig_sphere.add_trace(curve_trace(inverted_circle_points, "sphere inversion of C", "#ff7f0e", width=6))
fig_sphere.add_trace(curve_trace(inverted_rotated_points, "sphere inversion of rotated C", "#17becf", width=6))

for idx in [30, 70, 115, 160, 210]:
    fig_sphere.add_trace(
        segment_trace(
            np.array([sphere.center, line_points[idx], inverted_line_points[idx]]),
            f"reciprocal ray {idx}",
            "rgba(80,80,80,0.35)",
            width=2,
            marker_size=3,
        )
    )

fig_sphere.add_trace(point_trace(np.array([sphere.center, c1, c2, c3]), ["center", "c1", "c2", "c3"], "#111111"))
finish_figure(fig_sphere, "Chapter 1 variant: sphere inversion")

sphere_path = save_plotly_html(
    fig_sphere,
    "chapter-01",
    "plots",
    "sphere-inversion-variant.html",
    root=ARTIFACT_ROOT,
)
print(f"saved {sphere_path}")
fig_sphere


saved D:\Geometry\Geometric-Algebra-for-Computer-Science\artifacts\chapter-01\plots\sphere-inversion-variant.html


## Worked Example 3: The Reciprocal Distance Rule

Sphere inversion looks dramatic in the plot, but the numeric rule is tiny. For each sample
point, multiply the distance from the sphere center before inversion by the distance after
inversion. The product should be `r**2`.


In [9]:
sample_points = np.array([
    line_points[40],
    line_points[120],
    line_points[200],
    circle_points[25],
    rotated_circle_points[95],
])
sample_labels = ["line a", "line b", "line c", "circle", "rotated circle"]
sample_inverted = sphere.invert_points(sample_points)
d0 = np.linalg.norm(sample_points - sphere.center, axis=1)
d1 = np.linalg.norm(sample_inverted - sphere.center, axis=1)
products = d0 * d1

rows = ["| sample | before | after | product | product - r^2 |", "|---|---:|---:|---:|---:|"]
for label, before_d, after_d, product in zip(sample_labels, d0, d1, products):
    rows.append(f"| `{label}` | {before_d:.6f} | {after_d:.6f} | {product:.6f} | {product - sphere.radius**2:.2e} |")
Markdown("\n".join(rows))


| sample | before | after | product | product - r^2 |
|---|---:|---:|---:|---:|
| `line a` | 4.082244 | 0.307282 | 1.254400 | 2.22e-16 |
| `line b` | 2.100637 | 0.597152 | 1.254400 | 0.00e+00 |
| `line c` | 0.477202 | 2.628655 | 1.254400 | 2.22e-16 |
| `circle` | 1.214976 | 1.032449 | 1.254400 | -2.22e-16 |
| `rotated circle` | 1.328644 | 0.944121 | 1.254400 | -2.22e-16 |

## Programming Lesson

The coordinate code in this notebook still has special cases: one function reflects in a
plane, another inverts in a sphere, and a rotation matrix handles the line rotation. Chapter
1 argues that geometric algebra can move these ideas into a common product language.
That does not remove the need for implementation work, but it changes where the work
lives. Application code can talk about geometric objects and operators, while optimized
library code owns the product machinery.

The practical test is always the same: can you change the geometry without rewriting the
application around a new representation? The examples above are intentionally arranged
so that the same line, circle, and rotated circle can be sent through different operators.


### How this seed chapter maps to later chapters

The code in this notebook is intentionally modest, but every helper function is a placeholder for a larger algebraic operation introduced later.  The circle fitting routine is a coordinate version of an outer product construction: independent points span a higher-grade object, while dependent points collapse the grade and signal degeneracy.  The rotation example foreshadows rotors, where the transformation is represented by an even multivector and applied by a sandwich operation rather than by a matrix multiplied on one side.  The reflection example foreshadows versors, where a vector or blade representing the mirror determines the action.  The inversion example foreshadows conformal geometric algebra, where spheres and points become algebraic elements in a larger metric space.

This is why the chapter emphasizes checks more than formulas.  A good geometric computation should be testable by invariants: distances before and after a rotation, signed distances to a mirror plane, radius equality for points on a circle, or the product of original and inverted distances for a sphere inversion.  The notebooks that follow use the same habit.  Whenever a construction becomes more abstract, the final cell turns it back into something measurable so the algebra does not become decorative symbolism.

One practical lesson is that geometric algebra is not an alternative to numerical programming.  It is a way to organize numerical programming around geometric intent.  The final implementation still uses arrays, tolerances, plotting libraries, and tests.  What changes is the shape of the program: instead of storing a pile of special cases, the program stores elements of an algebra and lets products, duals, projections, and transformations expose the appropriate geometry.


## Sanity Checks

These checks make the notebook executable rather than merely illustrative.


In [10]:
# The three handles lie on the fitted circle and in its plane.
handle_radial_errors, handle_plane_errors = circle_point_errors(C, handle_points)
assert np.allclose(handle_radial_errors, 0.0, atol=1e-9)
assert np.allclose(handle_plane_errors, 0.0, atol=1e-9)

# Plane reflection changes the sign of signed distance to the plane.
original_distances = plane.signed_distance(circle_points)
reflected_distances = plane.signed_distance(reflected_circle_points)
reflection_error = np.max(np.abs(reflected_distances + original_distances))
assert np.allclose(reflected_distances, -original_distances, atol=1e-9)

# Plane reflection is an involution.
double_reflection_error = np.max(np.abs(plane.reflect_points(reflected_circle_points) - circle_points))
assert np.allclose(plane.reflect_points(reflected_circle_points), circle_points, atol=1e-9)

# Rotation about a line preserves every point's distance to that line.
original_axis_distances = L.distance_to_points(circle_points)
rotated_axis_distances = L.distance_to_points(rotated_circle_points)
rotation_error = np.max(np.abs(rotated_axis_distances - original_axis_distances))
assert np.allclose(rotated_axis_distances, original_axis_distances, atol=1e-9)

# Interpolation endpoints match the original and final rotated samples.
start_error = np.max(np.abs(rotated_at(0.0) - circle_points))
end_error = np.max(np.abs(rotated_at(1.0) - rotated_circle_points))
assert np.allclose(rotated_at(0.0), circle_points, atol=1e-9)
assert np.allclose(rotated_at(1.0), rotated_circle_points, atol=1e-9)

# Sphere inversion has the expected radius-distance product.
line_distance_before = np.linalg.norm(line_points - sphere.center, axis=1)
line_distance_after = np.linalg.norm(inverted_line_points - sphere.center, axis=1)
inversion_error = np.max(np.abs(line_distance_before * line_distance_after - sphere.radius**2))
assert np.allclose(line_distance_before * line_distance_after, sphere.radius**2, atol=1e-9)

artifact_paths = [route_path, construction_path, plane_path, sphere_path]
for artifact_path in artifact_paths:
    assert Path(artifact_path).exists(), artifact_path

print("sanity checks passed")
print(f"circle handle radial error: {np.max(np.abs(handle_radial_errors)):.2e}")
print(f"circle handle plane error: {np.max(np.abs(handle_plane_errors)):.2e}")
print(f"plane reflection signed-distance error: {reflection_error:.2e}")
print(f"double reflection error: {double_reflection_error:.2e}")
print(f"rotation axis-distance error: {rotation_error:.2e}")
print(f"rotation interpolation start/end errors: {start_error:.2e}, {end_error:.2e}")
print(f"sphere inversion radius-product error: {inversion_error:.2e}")
print("artifacts:")
for artifact_path in artifact_paths:
    print(f"- {artifact_path}")


sanity checks passed
circle handle radial error: 4.44e-16
circle handle plane error: 1.54e-16
plane reflection signed-distance error: 6.66e-16
double reflection error: 8.88e-16
rotation axis-distance error: 5.55e-16
rotation interpolation start/end errors: 2.22e-16, 0.00e+00
sphere inversion radius-product error: 4.44e-16
artifacts:
- D:\Geometry\Geometric-Algebra-for-Computer-Science\artifacts\chapter-01\plots\chapter-route-map.html
- D:\Geometry\Geometric-Algebra-for-Computer-Science\artifacts\chapter-01\plots\construction-handles.html
- D:\Geometry\Geometric-Algebra-for-Computer-Science\artifacts\chapter-01\plots\line-circle-plane-reflection.html
- D:\Geometry\Geometric-Algebra-for-Computer-Science\artifacts\chapter-01\plots\sphere-inversion-variant.html


## Chapter Takeaways

Chapter 1 is a promise, not yet the full machinery. The promise has three parts.

First, geometric objects should be first-class computational values. A circle, line, plane,
or sphere should not dissolve into unrelated coordinate bookkeeping as soon as a program
needs to transform it.

Second, transformations should act on those objects directly. The sandwich pattern is the
shape to watch: it lets one operator move or reflect a whole geometric entity.

Third, the rest of the book pays down the promise carefully. The next chapters introduce
products for spanning subspaces, measuring and projecting them, taking complements,
forming intersections, and finally using the geometric product as the unified operator
language hinted at here.
